# Case Study 2 — Stage 1: Label reconstruction from the official DEAP ratings

Student: Sanjeev Veeramani (Matr. 100004303) · Supervisor: Prof. Dr. Binh Vu

**Purpose.** A widely mirrored copy of the preprocessed DEAP files carries corrupted
valence and arousal columns while dominance and liking are intact. This notebook rebuilds
all labels from the official `participant_ratings.xls` distributed with DEAP, and verifies
that the rebuilt arrays match the label counts consumed by every later stage.

**Corruption signature observed in the mirrored copy** (see `01_deap_preprocessing_pipeline.ipynb`):

| | Mirrored copy | Official ratings |
|---|---|---|
| Valence mean | 3.12 | 5.25 |
| Valence range | 0.00 to 8.27 | 1.0 to 9.0 |
| High valence at threshold 5.0 | ~15% | 55.3% |

Dominance and liking in the mirrored copy match the official file, which is what localises
the problem to the valence and arousal columns rather than to the loading pipeline.

**Input.** `participant_ratings.xls` is part of the DEAP distribution and is obtained under
the DEAP end-user licence agreement. It is not redistributed in this repository.

**Outputs.** Per-window label arrays used from stage 2 onward:
`y_deap_valence.npy`, `y_deap_arousal.npy`, `y_deap_3class.npy`.

In [1]:
import pandas as pd, numpy as np

RATINGS = 'participant_ratings.xls'   # official DEAP metadata, obtained under the DEAP EULA
WINDOWS_PER_TRIAL = 119               # 60 s of signal, 1 s windows, 50% overlap, after baseline removal
T_BIN, T_HIGH, T_LOW = 5.0, 5.5, 4.5

df = pd.read_excel(RATINGS).sort_values(['Participant_id', 'Experiment_id']).reset_index(drop=True)
# NOTE: the preprocessed DEAP .dat files store trials in video (Experiment_id) order,
# not presentation (Trial) order. Sorting by Trial gives the same class counts but a
# different trial-to-label assignment.
print(f'rows: {len(df)}   subjects: {df.Participant_id.nunique()}   trials per subject: {len(df)//df.Participant_id.nunique()}')

v, a = df['Valence'].values, df['Arousal'].values
d, l = df['Dominance'].values, df['Liking'].values

print('\nOfficial rating distributions:')
for name, x in [('Valence', v), ('Arousal', a), ('Dominance', d), ('Liking', l)]:
    print(f'  {name:10s} mean={x.mean():.2f}  range=[{x.min():.1f}, {x.max():.1f}]  '
          f'above {T_BIN}: {(x > T_BIN).sum()} ({100*(x > T_BIN).mean():.1f}%)')

val_bin = (v > T_BIN).astype(np.int64)
aro_bin = (a > T_BIN).astype(np.int64)
val_3c  = np.where(v > T_HIGH, 2, np.where(v < T_LOW, 0, 1)).astype(np.int64)

print('\nPer-trial label counts:')
print('  binary valence [low, high] :', np.bincount(val_bin))
print('  binary arousal [low, high] :', np.bincount(aro_bin))
print('  3-class valence [neg, neu, pos] :', np.bincount(val_3c))

val_bin_w = np.repeat(val_bin, WINDOWS_PER_TRIAL)
aro_bin_w = np.repeat(aro_bin, WINDOWS_PER_TRIAL)
val_3c_w  = np.repeat(val_3c,  WINDOWS_PER_TRIAL)

print(f'\nPer-window expansion ({WINDOWS_PER_TRIAL} windows per trial):')
print('  total windows :', len(val_bin_w))
print('  binary valence [low, high] :', np.bincount(val_bin_w))
print('  binary arousal [low, high] :', np.bincount(aro_bin_w))
print('  3-class valence [neg, neu, pos] :', np.bincount(val_3c_w))

print('\nCross-check against the arrays consumed downstream:')
checks = [
    ('per-trial binary valence',  tuple(np.bincount(val_bin)),   (572, 708)),
    ('per-trial binary arousal',  tuple(np.bincount(aro_bin)),   (543, 737)),
    ('per-trial 3-class valence', tuple(np.bincount(val_3c)),    (472, 221, 587)),
    ('per-window binary valence', tuple(np.bincount(val_bin_w)), (68068, 84252)),
]
for name, got, expected in checks:
    print(f'  {name:26s} {got}  expected {expected}   {"OK" if got == expected else "MISMATCH"}')

rows: 1280   subjects: 32   trials per subject: 40

Official rating distributions:
  Valence    mean=5.25  range=[1.0, 9.0]  above 5.0: 708 (55.3%)
  Arousal    mean=5.16  range=[1.0, 9.0]  above 5.0: 737 (57.6%)
  Dominance  mean=5.38  range=[1.0, 9.0]  above 5.0: 780 (60.9%)
  Liking     mean=5.52  range=[1.0, 9.0]  above 5.0: 851 (66.5%)

Per-trial label counts:
  binary valence [low, high] : [572 708]
  binary arousal [low, high] : [543 737]
  3-class valence [neg, neu, pos] : [472 221 587]

Per-window expansion (119 windows per trial):
  total windows : 152320
  binary valence [low, high] : [68068 84252]
  binary arousal [low, high] : [64617 87703]
  3-class valence [neg, neu, pos] : [56168 26299 69853]

Cross-check against the arrays consumed downstream:
  per-trial binary valence   (np.int64(572), np.int64(708))  expected (572, 708)   OK
  per-trial binary arousal   (np.int64(543), np.int64(737))  expected (543, 737)   OK
  per-trial 3-class valence  (np.int64(472), np.int64(221

## Verification

The cross-checks above tie this notebook to the rest of the study:

- the per-trial counts match the label distributions printed in `04_multi_source_da/02_per_trial_protocol.ipynb`;
- the per-window counts match the class distributions printed in `02_classical_baselines/02_svm_random_forest.ipynb`
  (`valence {0: 68068, 1: 84252}`, `arousal {0: 64617, 1: 87703}`, `3class {0: 56168, 1: 26299, 2: 69853}`).

A 55.3 / 44.7 valence split is recovered, against the roughly 15 / 85 split produced by the mirrored labels.

The cell below compares the rebuilt array element by element against the `y_deap_valence.npy`
committed from the original run. Class counts alone are not sufficient here: sorting the ratings
by `Trial` rather than by `Experiment_id` produces identical counts but a different trial-to-label
assignment, so the byte-level check is what establishes that the ordering is also correct.

In [2]:
# Byte-level check against the label array used in the experiments
# Point this at the y_deap_valence.npy produced by the original run.
COMMITTED = 'processed/y_deap_valence.npy'

committed = np.load(COMMITTED)
print('Byte-level check against', COMMITTED)
print('  shapes match      :', committed.shape == val_bin_w.shape)
print('  arrays identical  :', np.array_equal(committed, val_bin_w))

Byte-level check against processed/y_deap_valence.npy
  shapes match      : True
  arrays identical  : True


In [3]:
import numpy as np
from pathlib import Path

OUT = Path('processed'); OUT.mkdir(exist_ok=True)
np.save(OUT / 'y_deap_valence.npy', val_bin_w)
np.save(OUT / 'y_deap_arousal.npy', aro_bin_w)
np.save(OUT / 'y_deap_3class.npy',  val_3c_w)
print('saved:', [p.name for p in sorted(OUT.glob('y_deap_*.npy'))])
for name in ['y_deap_valence', 'y_deap_arousal', 'y_deap_3class']:
    arr = np.load(OUT / f'{name}.npy')
    print(f'  {name:16s} shape={arr.shape}  dtype={arr.dtype}  counts={np.bincount(arr)}')

saved: ['y_deap_3class.npy', 'y_deap_arousal.npy', 'y_deap_valence.npy']
  y_deap_valence   shape=(152320,)  dtype=int64  counts=[68068 84252]
  y_deap_arousal   shape=(152320,)  dtype=int64  counts=[64617 87703]
  y_deap_3class    shape=(152320,)  dtype=int64  counts=[56168 26299 69853]
